## Task 1: Regression network training and testing with PyTorch Lightning

#### Goal: Implement a regression MLP using PyTorch Lightning on a toy dataset.

1. Install the required libraries: `pip install pytorch-lightning` and `pip install ray[tune]`.
2. Implement a `Dataset` subclass (`MyDataset`) that stores input features and targets and returns individual samples via `__getitem__`.
3. Implement a `LightningDataModule` (`MyDataModule`) that generates a regression toy dataset using `make_regression`, converts it to `float32` tensors, splits it into train/val/test subsets, and exposes them via `train_dataloader`, `val_dataloader`, and `test_dataloader`.
4. Implement a `LightningModule` (`LitModel`) with a 3-layer MLP. Use `self.save_hyperparameters()` in `__init__`. Implement `training_step`, `validation_step`, and `test_step` — each should compute MSE loss; validation and test steps should also compute and log R² score. Implement `configure_optimizers` returning an Adam optimizer using `self.hparams.lr`.
5. Instantiate a `Trainer` with `max_epochs` and appropriate `accelerator` settings and run `trainer.fit()` followed by `trainer.test()`.
6. Add an `EarlyStopping` callback (monitoring `val_loss` with patience 5) and a `ModelCheckpoint` callback (saving the top 3 checkpoints by `val_loss` plus the last one). Re-run training with both callbacks active.
7. Define a `train_model(config)` function for Ray Tune that instantiates `LitModel` with sampled `hidden_size` and `lr` values and trains it using a `TuneReportCallback`. Run the hyperparameter search using `tune.run()` with `tune.choice` for `hidden_size` and `tune.loguniform` for `lr` over 10 samples, then print the best config.

**Assignment:** Implement and run hyperparameter optimization for a regression network on the toy dataset using PyTorch Lightning.


In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
from utils import *
import torch

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [17]:
import pytorch_lightning as pl

n_features = 12

datamodule = MyDataModule(batch_size = 32, n_samples = 8000, n_features = n_features)
model = LitModel(hidden_layers = [64, 256, 64], input_dim = n_features, output_dim = 1)

trainer = pl.Trainer(max_epochs=10, accelerator=device, devices=1)
trainer.fit(model, datamodule=datamodule)
trainer.test(model, datamodule=datamodule)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name          | Type       | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | input_layer   | Linear     | 832    | train | 0    
1 | hidden_layers | ModuleList | 33.1 K | train | 0    
2 | output_layer  | Linear     | 65     | train | 0    
3 | activation    | ReLU       | 0      | train | 0    
4 | criterion     | MSELoss    | 0      | train | 0    
-------------------------------------------------------------
34.0 K    Trainable param

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           21.042949676513672
         test_r2            0.9990431070327759
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 21.042949676513672, 'test_r2': 0.9990431070327759}]